In [79]:
import networkx as nx
import pandas as pd
import pickle
from collections import defaultdict
import ast
import re

### Convert KG to (head[tab]rel[tab]tail) format for Anyburl

+ remove or merge non-causal relations
+ remove AD-node to mine rules

In [3]:
# process KG to remove non-causal relations and remove AD node
kg_path = "../data/KG/ad_network_with_reverse_edges.pkl"
non_causal_relations = ['positive_correlation',
    'association',
    'causes_no_change',
    'negative_correlation']

with open(kg_path,"rb")as f:
    G = pickle.load(f)

def change_kg(G, non_causal_relations:list, output:str, causal_only=True, no_AD=True):
    
    # remove non_causal relations
    unique_rels = set()   
    edges_to_remove = [] 
    for u,v,rel,data in G.edges(data=True, keys=True):
        unique_rels.add(rel)
        if rel in non_causal_relations:
            edges_to_remove.append((u,v,rel))
    if causal_only == True:
        G.remove_edges_from(edges_to_remove)
        # save causal-only-KG
        with open(output, 'wb') as f:
            pickle.dump(G, f)
    if no_AD: 
        # remove AD node
        for n in G.nodes():
            if "Alzheimer Disease" in n:
                ad_node = n
        G.remove_node(ad_node)

        # save causal-only-KG
        with open(output, 'wb') as f:
            pickle.dump(G, f)
    return G
        

In [10]:
def get_anyburl_df(G:nx.MultiDiGraph, non_causal_relations=None, health_kg=True, output = 'train.txt'):

    triples = []
    i = 0
    if health_kg:
        for u, v, rel, attr in G.edges(data=True, keys=True):
            u_name = G.nodes[u]['bel']
            v_name = G.nodes[v]['bel']
            rel_type = attr.get('type')
            triples.append([u_name, rel_type, v_name])
    elif non_causal_relations is not None:
        for u, v, rel, data in G.edges(data=True, keys=True): 
            if 'rev_' not in rel and rel not in non_causal_relations:
                triples.append([u, rel, v])
    else:
        for u, v, rel, data in G.edges(data=True, keys=True): 
            if 'rev_' not in rel:
                triples.append([u, rel, v])
                    
    df = pd.DataFrame(triples)
    df.to_csv(output, sep='\t', header=False, index=False)
    return df

In [11]:
# get AnyBURL train.txt for causal_relation.txt no_ad_causal.txt
with open('../data/KG/no_ad_causal_only.pkl', 'rb') as f:
    G_no_ad = pickle.load(f)
with open('../data/KG/ad_causal_only.pkl','rb') as f:
    G_causal = pickle.load(f)
with open('../data/KG/healthy_aging_kg.pkl', 'rb') as f:
    healthy_kg = pickle.load(f)

# convert to train.txt file
df_all = get_anyburl_df(G=G, health_kg=False,output='../data/train.txt')
df_no_ad = get_anyburl_df(G_no_ad,non_causal_relations,False,'../data/train_no_ad.txt')
df_causal = get_anyburl_df(G_causal, non_causal_relations, False, '../data/train_causal.txt')
df_health = get_anyburl_df(G=healthy_kg,health_kg=True, output='../data/train_healthy.txt')

In [12]:
df_all

,0,1,2
0,"g(HGNC:""BDNF"")",decreases,"p(HGNC:""APP"",frag(""672_713""))"
1,"g(HGNC:""BDNF"")",increases,"g(HGNC:""SORL1"")"
2,"g(HGNC:""BDNF"")",increases,"p(HGNC:""BDNF"")"
3,"g(HGNC:""PPP1CC"")",association,"path(MESH:""Alzheimer Disease"")"
4,"g(HGNC:""PPP1CC"")",association,"bp(MESH:""Polymorphism, Single Nucleotide"")"
...,...,...,...
7851,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""NR4A2"")"
7852,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""ADAM10"")"
7853,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""FOS"")"
7854,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""SIRT1"")"


In [14]:
df_no_ad

,0,1,2
0,"g(HGNC:""BDNF"")",decreases,"p(HGNC:""APP"",frag(""672_713""))"
1,"g(HGNC:""BDNF"")",increases,"g(HGNC:""SORL1"")"
2,"g(HGNC:""BDNF"")",increases,"p(HGNC:""BDNF"")"
3,"g(HGNC:""VLDLR"")",increases,"bp(GO:""lipid metabolic process"")"
4,"g(HGNC:""VLDLR"")",increases,"bp(GO:""central nervous system development"")"
...,...,...,...
5070,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(FPLX:""CREB"")"
5071,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""NR4A2"")"
5072,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""ADAM10"")"
5073,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""FOS"")"


In [13]:
df_causal

,0,1,2
0,"g(HGNC:""BDNF"")",decreases,"p(HGNC:""APP"",frag(""672_713""))"
1,"g(HGNC:""BDNF"")",increases,"g(HGNC:""SORL1"")"
2,"g(HGNC:""BDNF"")",increases,"p(HGNC:""BDNF"")"
3,"g(HGNC:""VLDLR"")",increases,"bp(GO:""lipid metabolic process"")"
4,"g(HGNC:""VLDLR"")",increases,"bp(GO:""central nervous system development"")"
...,...,...,...
5272,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(FPLX:""CREB"")"
5273,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""NR4A2"")"
5274,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""ADAM10"")"
5275,"composite(a(CHEBI:""Ibotenic acid""),p(HGNC:""APP...",decreases,"p(HGNC:""FOS"")"


In [15]:
df_health

,0,1,2
0,"r(MGI:""Bace1"")",regulates,"a(CHEBI:""amyloid-beta"")"
1,"a(CHEBI:""amyloid-beta"")",regulates,"bp(GOBP:""lipid homeostasis"")"
2,"a(CHEBI:""amyloid-beta"")",regulates,"bp(GOBP:""memory"")"
3,"a(CHEBI:""amyloid-beta"")",regulates,"bp(GOBP:""memory"")"
4,"a(CHEBI:""amyloid-beta"")",regulates,"bp(GOBP:""memory"")"
...,...,...,...
7405,"g(HGNCGENEGROUP:""Glutamate metabotropic recept...",regulates,"bp(GOBP:""long-term synaptic depression"")"
7406,"bp(PW:""ion homeostasis pathway"")",regulates,"bp(GOBP:""cell death"")"
7407,"bp(PW:""kinase mediated signaling pathway"")",regulates,"bp(GOBP:""learning"")"
7408,"bp(PW:""kinase mediated signaling pathway"")",regulates,"bp(GOBP:""memory"")"


### Compare rules between files

In [44]:
# Define the scoring hierarchy
scores_map = {
    'directly_increases': 1.0,
    'increases': 0.8,
    'positive_correlation': 0.5,
    'association': 0.1,
    'causes_no_change': 0.0,
    'negative_correlation': -0.5,
    'decreases': -0.8,
    'directly_decreases': -1.0
}

In [ ]:
def load_rules(filepath):
    """Returns a set of the logic strings and a dict mapping logic to confidence."""
    rules_logic = set()
    rule_data = {}

    rules_logic = set()
    rule_data = {}
    with open(filepath, 'r') as f:
        for line in f:
            if "<=" not in line: continue  # Skip headers
            parts = line.strip().split("\t")
            # AnyBURL format: [Predicted_Count] [Correct_Count] [Confidence] [Rule_Logic]
            conf = float(parts[2])
            logic = parts[3]
            rules_logic.add(logic)
            rules_data[logic] = conf
            # # filter for AD
            # if "Alzheimer Disease" in logic and 'p(HGNC' in logic:
            #     rules_logic.add(logic)
            #     rule_data[logic] = conf
            # if "Alzheimer Disease" in logic and 'g(HGNC' in logic:
            #     rules_logic.add(logic)
            #     rule_data[logic]=conf
    return rules_logic, rule_data

### NO-AD-Graph mined rules

Get ad-linked nodes: the nodes that have direct causal edge to ad-node 

In [ ]:
ad_linked_df = pd.read_csv("../data/ad_linked_nodes.csv", sep='\t')
increase_ad_df = ad_linked_df[ad_linked_df['Relation'].str.contains('increases', na=False)]
decrease_ad_df = ad_linked_df[ad_linked_df['Relation'].str.contains('decreases', na=False)]

increase_ad_nodes = list(increase_ad_df['Node'])
decrease_ad_nodes = list(decrease_ad_df['Node'])
print(f'{len(increase_ad_nodes)} nodes that increase AD:\n', increase_ad_nodes, f'\n\n{len(decrease_ad_nodes)} nodes that decrease AD:\n', decrease_ad_nodes)

26 nodes that increase AD:
 ['g(HGNC:"CD33")', 'g(HGNC:"RFC1",var("G,80,A"))', 'g(HGNC:"CR1")', 'g(HGNC:"MTHFR",var("C,677,T"))', 'g(HGNC:"CLU")', 'g(HGNC:"PICALM")', 'p(HGNC:"APP",var("N,10,Y"))', 'p(HGNC:"APP")', 'p(HGNC:"APP",var("A,713,T"))', 'p(HGNC:"PSEN1")', 'p(HGNC:"H3F3A",pmod(Ub))', 'p(HGNC:"H3F3A",var("P,T,231"))', 'p(HGNC:"MTHFR",var("A,677,V"))', 'p(HGNC:"APP",frag("1_8"))', 'p(HGNC:"APP",frag("9_16"))', 'p(HGNC:"LRP8")', 'p(HGNC:"DYRK1A")', 'p(HGNC:"MMP2")', 'p(HGNC:"GRN")', 'p(HGNC:"APP",frag("1_16"))', 'p(HGNC:"BACE1")', 'p(HGNC:"AGER")', 'p(HGNC:"MAPT",pmod(Ph))', 'p(HGNC:"S100A9")', 'p(HGNC:"APP",frag("672_713"))', 'p(HGNC:"LRP1")'] 

4 nodes that decrease AD:
 ['g(HGNC:"APP",var("A,673,T"))', 'p(HGNC:"IL1B")', 'p(HGNC:"PLAT")', 'p(HGNC:"CCR5")']


analysis on rules mined from No-AD-KG

In [16]:
column_names = ["Predicted_Count", "Correct_Count", "Confidence", "Rule_Logic"]
df_nad_100 = pd.read_csv("../results/no_ad/rules.txt-100", sep='\t', header=0, names=column_names)
df_nad_300 = pd.read_csv("../results/no_ad/rules.txt-300", sep='\t',header=0, names=column_names)
df_nad_600 = pd.read_csv("../results/no_ad/rules.txt-600", sep='\t',header=0, names=column_names)

## Extract rules containing increase_ad nodes

In [91]:
def if_contains(row_string, match_list):
    replaced = row_string.replace('""','"')
    for node in match_list:
        if node in replaced:
            return True
        
    return False

def extract_target_rules(df, match_list, output, confidence_threshold = 0.5, correct_count_threshold=2):
    df = df.sort_values(by=['Confidence','Correct_Count'], ascending=[False,False])
    filtered_df = df[(df['Confidence']>=confidence_threshold) & (df_nad_100['Correct_Count']>correct_count_threshold)]
    increase_ad_rules = filtered_df[filtered_df['Rule_Logic'].apply(lambda x: if_contains(str(x), match_list))]
    increase_ad_rules.to_csv(output, sep='\t')
    print(f"Found {len(increase_ad_rules)} target rules")
    return increase_ad_rules
df_nad_increase_100 = extract_target_rules(df_nad_100, increase_ad_nodes, 'nad_100_rules.txt')
for i in df_nad_increase_100['Rule_Logic']:
    print(i)

Found 82 target rules
increases("p(HGNC:""PSEN1"")",Y) <= increases("p(HGNC:""PSEN2"")",Y)
increases(X,"p(HGNC:""APP"",frag(""672_713""))") <= increases(X,"p(HGNC:""APP"",frag(""672_711""))")
increases(X,"deg(p(HGNC:""APP"",frag(""672_713"")))") <= increases(X,"deg(p(HGNC:""APP"",frag(""672_711"")))")
increases("p(HGNC:""PSEN1"")",Y) <= decreases("p(HGNC:""PSEN1"",var(""E,280,A""))",Y)
increases("act(p(HGNC:""PSEN1""))",Y) <= increases("act(p(HGNC:""PSEN2""))",Y)
increases("act(p(HGNC:""PSEN2""))",Y) <= increases("act(p(HGNC:""PSEN1""))",Y)
increases("p(HGNC:""PSEN1"")",Y) <= decreases("p(HGNC:""PSEN2"",var(""N,141,I""))",Y)
increases("p(HGNC:""PSEN1"")",Y) <= increases("p(HGNC:""PSEN1"",var(""E,280,A""))",Y)
increases("p(HGNC:""NFKB2"")",Y) <= increases("composite(p(HGNC:""APP"",frag(""672_713"")),p(HGNC:""C1QBP""))",Y)
increases("p(HGNC:""IL10"")",Y) <= increases("composite(p(HGNC:""APP"",frag(""672_713"")),p(HGNC:""C1QBP""))",Y)
increases("p(HGNC:""PSEN1"")",Y) <= increases("p(HGNC:

In [92]:
df_nad_increase_300 = extract_target_rules(df_nad_300, increase_ad_nodes, 'nad_300_rules.txt')
for i in df_nad_increase_300['Rule_Logic']:
    print(i)

Found 184 target rules
increases(X,"p(HGNC:""APP"",frag(""672_713""))") <= increases(X,"p(HGNC:""APP"",frag(""672_711""))")
increases("p(HGNC:""PSEN1"")",Y) <= decreases("p(HGNC:""PSEN2"",var(""N,141,I""))",Y)
increases("p(HGNC:""NFKB2"")",Y) <= increases("composite(p(HGNC:""APP"",frag(""672_713"")),p(HGNC:""C1QBP""))",Y)
increases(X,"deg(p(HGNC:""APP""))") <= decreases("act(p(HGNC:""GRIN2B""))",X)
increases("p(HGNC:""PSEN1"")",Y) <= increases(Y,"p(HGNC:""BCL2L11"")")
increases(X,"deg(p(HGNC:""APP""))") <= directly_increases(X,"rxn(reactants(p(HGNC:""APP"")),products(a(CHEBI:""amyloid-beta"")))")
increases("p(HGNC:""APP"",frag(""672_713""))",Y) <= decreases("p(HGNC:""GSK3B"",var(""Ph,S,9""))",Y)
increases(X,"act(p(HGNC:""MAPK1""))") <= increases("complex(p(HGNC:""APP""),p(HGNC:""GRB2""))",X)
increases("composite(p(HGNC:""APP"",frag(""672_713"")),p(HGNC:""C1QBP""))",Y) <= decreases("a(MESH:""Cholinesterase Inhibitors"")",Y)
increases("complex(p(HGNC:""GRB2""),p(HGNC:""SHC2""))",Y) <= in

/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_12191/2015677745.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df = df[(df['Confidence']>=confidence_threshold) & (df_nad_100['Correct_Count']>correct_count_threshold)]


In [93]:
df_nad_increase_600 = extract_target_rules(df_nad_600, increase_ad_nodes, 'nad_600_rules.txt')
for i in df_nad_increase_600['Rule_Logic']:
    print(i)

Found 187 target rules
increases("p(HGNC:""PSEN1"")",Y) <= increases("p(HGNC:""PSEN2"")",Y)
increases(X,"deg(p(HGNC:""APP"",frag(""672_713"")))") <= increases(X,"deg(p(HGNC:""APP"",frag(""672_711"")))")
increases(X,"deg(p(HGNC:""APP""))") <= decreases("act(p(HGNC:""GRIN2B""))",X)
increases(X,"deg(p(HGNC:""APP""))") <= decreases("act(p(HGNC:""GRIN3B""))",X)
increases("p(HGNC:""APP"",frag(""672_713""))",Y) <= decreases("p(HGNC:""GSK3B"",var(""Ph,S,9""))",Y)
increases(X,"deg(p(HGNC:""APP""))") <= decreases("act(p(HGNC:""GRIN2C""))",X)
increases(X,"p(HGNC:""MAPT"",pmod(Ph))") <= decreases("p(HGNC:""A2M"")",X)
increases(X,"p(HGNC:""APP"",frag(""672_713""))") <= increases("p(HGNC:""ABCA2"")",X)
increases(X,"p(HGNC:""APP"")") <= decreases("a(CHEBI:""thyroid hormone"")",X)
increases("p(HGNC:""APP"",var(""P,T,668""))",Y) <= decreases("complex(p(HGNC:""APBA1""),p(HGNC:""APP""))",Y)
increases("p(HGNC:""BACE1"")",Y) <= increases(Y,"act(p(HGNC:""MSR1""))")
increases("p(HGNC:""APP"")",Y) <= increase

/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_12191/2015677745.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_df = df[(df['Confidence']>=confidence_threshold) & (df_nad_100['Correct_Count']>correct_count_threshold)]


check if rule-body's end node matches with ad-linked-nodes

In [84]:
def extract_and_match(row_string, match_list):
    # to find the part between the first comma and the ') <='
    pattern = r',\s*(.+)\)\s*<='
    match = re.search(pattern, row_string)
    
    if match:
        extracted = match.group(1)
        #print(extracted)
        # Normalize the string: change "" to " to match your list format
        normalized = extracted.replace('""', '"')
        print(normalized)
        
        if normalized in match_list:
            return True
    return False

# Filtering your dataframe
# Assuming 'Rule' is the column name
filtered_df = df_nad_100[df_nad_100['Rule_Logic'].apply(lambda x: extract_and_match(str(x), increase_ad_nodes))]

# Display results
print(filtered_df)

p(HGNC:"PRKCA"))",Y
"a(CHEBI:"glutamate(1-)")"
Y
Y
Y
Y
"a(CHEBI:"glutamate(1-)")"
p(HGNC:"PRKCA"))",Y
Y
Y
"p(HGNC:"MAPT",var("P,T,205"))"
"p(HGNC:"APP",frag("672_713"))"
ma(kin))",Y
"p(HGNC:"MAPT",var("P,S,202"))"
"deg(p(HGNC:"APP",frag("672_713")))"
"p(HGNC:"MAPT",var("P,S,202"))"
Y
"p(HGNC:"MAPT",var("P,T,205"))"
"act(p(FPLX:"ERK"))"
"act(p(HGNC:"CHRNA5"))"
"act(p(HGNC:"CHRNA2"))"
"act(p(HGNC:"CHRNA2"))"
"act(p(FPLX:"ERK"))"
"act(p(HGNC:"CHRNB2"))"
Y
a(MESH:"Pituitary Gland")))",Y
"act(p(HGNC:"CHRNA10"))"
"act(p(HGNC:"CHRNA3"))"
"act(p(HGNC:"CHRNA9"))"
"act(p(HGNC:"CHRNA10"))"
"act(p(HGNC:"CHRNA4"))"
"p(HGNC:"MAPT",var("P,S,396"))"
"act(p(HGNC:"CHRNB3"))"
Y
"act(p(HGNC:"CHRNB2"))"
Y
"act(p(HGNC:"CHRNA9"))"
"act(p(HGNC:"PDPK1"))"
"act(p(HGNC:"CHRNB3"))"
"p(HGNC:"SOS1")"
Y
"act(p(HGNC:"CHRNA5"))"
"act(p(HGNC:"CHRNA6"))"
"act(p(HGNC:"CHRNA3"))"
Y
"act(p(HGNC:"CHRNA2"))"
Y
"act(p(HGNC:"CHRNA6"))"
Y
"act(p(HGNC:"CHRNA6"))"
"act(p(HGNC:"CHRNA6"))"
"p(HGNC:"SOS2")"
"act(p(HGNC:"CHRNA2"))"
"

### Causal-Only-Graph mined rules

### Origin-Graph mined rules